# model_experiment_LightGBM

Global gradient-boosting model over all 3331 series. Direct multi-horizon: every lag feature is >= 39 weeks back, so one `predict()` covers the whole 39-week test set with no recursion.

MLflow experiment: **LightGBM_Training** — runs: Cleaning, CV_*, Feature_Importance, Final.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # repo root on path

import numpy as np
import pandas as pd
import mlflow

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow

train, test, features, stores = load_raw()
print(train.shape, test.shape)

from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

from src.preprocessing import WalmartPreprocessor, make_xyw
from src.postprocess import apply_christmas_shift
from src.mlflow_utils import REGISTRY_MODEL_NAME

setup_mlflow("LightGBM_Training")

## Run 1 — Cleaning (documenting the data decisions)

In [ ]:
with mlflow.start_run(run_name="LightGBM_Cleaning"):
    mlflow.log_params({
        "negative_sales": "kept (returns are real; metric is L1 on raw dollars)",
        "markdown_na": "fill 0 + <col>_present flag (NA = no promo running)",
        "cpi_unemployment_na": "forward-fill per store (covers test tail)",
        "target_transform": "none (L1 objective on raw sales matches WMAE)",
        "n_rows": len(train),
        "n_series": train.groupby(["Store", "Dept"]).ngroups,
    })

## Run 2 — Cross-validation on the shared folds

`sample_weight = 1 + 4*IsHoliday` so training optimizes what Kaggle measures.

In [ ]:
def eval_fold(params, fold):
    tr, va = split_fold(train, fold)
    Xtr, ytr, wtr = make_xyw(tr)
    Xva, yva, _ = make_xyw(va)
    prep = WalmartPreprocessor(features_df=features, stores_df=stores)
    Xtr_t = prep.fit(Xtr, ytr).transform(Xtr)
    Xva_t = prep.transform(Xva)
    model = LGBMRegressor(**params)
    model.fit(Xtr_t, ytr, sample_weight=wtr)
    rep = wmae_report(yva, model.predict(Xva_t), va["IsHoliday"])
    return rep, model, Xtr_t

In [ ]:
PARAM_GRID = [
    dict(objective="l1", n_estimators=800, learning_rate=0.05, num_leaves=127,
         min_child_samples=30, colsample_bytree=0.8, random_state=42, verbose=-1),
    dict(objective="l1", n_estimators=1500, learning_rate=0.03, num_leaves=255,
         min_child_samples=20, colsample_bytree=0.8, random_state=42, verbose=-1),
    dict(objective="l1", n_estimators=800, learning_rate=0.05, num_leaves=63,
         min_child_samples=50, colsample_bytree=0.9, random_state=42, verbose=-1),
]

results = []
for i, params in enumerate(PARAM_GRID):
    with mlflow.start_run(run_name=f"LightGBM_CV_{i}"):
        mlflow.log_params({**params, "fold": 1})
        rep, model, _ = eval_fold(params, fold=1)
        mlflow.log_metrics(rep)
    results.append((i, rep["wmae"]))
    print(i, rep)
best_i = min(results, key=lambda r: r[1])[0]
print("best config:", best_i)

In [ ]:
# sanity-check the winner on Fold 2 (regular weeks, no big holidays)
with mlflow.start_run(run_name="LightGBM_CV_best_fold2"):
    mlflow.log_params({**PARAM_GRID[best_i], "fold": 2})
    rep2, _, _ = eval_fold(PARAM_GRID[best_i], fold=2)
    mlflow.log_metrics(rep2)
print(rep2)

## Run 3 — Feature importance

In [ ]:
rep, model, Xtr_t = eval_fold(PARAM_GRID[best_i], fold=1)
imp = pd.Series(model.feature_importances_, index=Xtr_t.columns).sort_values(ascending=False)
with mlflow.start_run(run_name="LightGBM_Feature_Importance"):
    mlflow.log_dict(imp.to_dict(), "feature_importance.json")
    mlflow.log_metrics(rep)
print(imp.head(20))

## Run 4 — Final: fit Pipeline on ALL train, log to MLflow, predict raw test

The pipeline (preprocessing + model in one object) is what gets registered — it runs directly on the un-preprocessed test set.

In [ ]:
BEST = PARAM_GRID[best_i]
X, y, w = make_xyw(train)
pipe = Pipeline([
    ("prep", WalmartPreprocessor(features_df=features, stores_df=stores)),
    ("model", LGBMRegressor(**BEST)),
])
pipe.fit(X, y, model__sample_weight=w)

with mlflow.start_run(run_name="LightGBM_Final"):
    mlflow.log_params(BEST)
    mlflow.log_metric("fold1_wmae", min(r[1] for r in results))
    # if LightGBM ends up the overall best architecture, add:
    #   registered_model_name=REGISTRY_MODEL_NAME
    mlflow.sklearn.log_model(pipe, name="model",
                             registered_model_name="walmart-lightgbm")

In [ ]:
# raw test in -> submission out
raw_test = test[["Store", "Dept", "Date"]]
sub = test[["Store", "Dept", "Date"]].copy()
sub["pred"] = pipe.predict(raw_test)
sub = apply_christmas_shift(sub, pred_col="pred")
make_submission(sub, "pred", "submission_lightgbm.csv")
print("wrote submission_lightgbm.csv")